In [1]:
import pandas as pd

data = pd.read_csv('../Dataset/Scored_Cleaned_Data.csv')

In [2]:
import numpy as np
import pandas as pd
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score
from sklearn.model_selection import KFold

def evaluate_gmm_custom_logic(df, target_col='Risk_score', n_folds=5):
    X_df = df.drop(columns=["Timestamp", "Group"], errors='ignore')
    spend_values = df[target_col].values
    
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)
    overall_results = []

    print(f"{'K':<5} | {'Avg Silhouette':<15} | {'Custom Deviation':<20}")
    print("-" * 50)

    for k in range(3, 20):
        fold_silhouettes = []
        fold_custom_deviations = []
        
        for train_idx, val_idx in kf.split(X_df):
            X_fold = X_df.iloc[train_idx]
            spend_fold = spend_values[train_idx]
            
            model = GaussianMixture(n_components=k, random_state=42)
            labels = model.fit_predict(X_fold)
            
            # Silhouette
            sil = silhouette_score(X_fold, labels)
            fold_silhouettes.append(sil)
            
            # (sum(|M - means|^0.5))^2
            cluster_means = [spend_fold[labels == cid].mean() for cid in np.unique(labels) if np.any(labels == cid)]
            c_means = np.array(cluster_means)
            M = np.mean(c_means)
            diffs = np.abs(M - c_means)
            custom_dev = (np.sum(diffs**0.5))**2
            fold_custom_deviations.append(custom_dev)

        avg_sil = np.mean(fold_silhouettes)
        avg_dev = np.mean(fold_custom_deviations)
        overall_results.append({'k': k, 'Silhouette': avg_sil, 'Custom_Deviation': avg_dev})
        print(f"{k:<5} | {avg_sil:<15.4f} | {avg_dev:<20.2f}")

    return pd.DataFrame(overall_results)

gmm_custom_results = evaluate_gmm_custom_logic(data)

K     | Avg Silhouette  | Custom Deviation    
--------------------------------------------------
3     | -0.0436         | 117.39              
4     | -0.1385         | 147.57              
5     | -0.1192         | 404.39              
6     | -0.1288         | 658.64              
7     | -0.1269         | 1230.31             
8     | -0.0030         | 1751.88             
9     | 0.0226          | 2417.06             
10    | 0.1165          | 3204.51             
11    | 0.1788          | 3782.89             
12    | 0.1912          | 4676.70             
13    | 0.2224          | 5809.69             
14    | 0.2626          | 6612.69             
15    | 0.2687          | 7181.99             
16    | 0.2807          | 7938.63             
17    | 0.2718          | 8697.56             
18    | 0.2607          | 9218.50             
19    | 0.3125          | 10254.49            
